In [4]:
import requests
teams = requests.get("https://api-web.nhle.com/v1/standings/2026-04-17")

In [5]:
teams

<Response [200]>

In [10]:
data: dict = teams.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['wildCardIndicator', 'standingsDateTimeUtc', 'standings'])


In [7]:
type(data)

dict

In [8]:
data

{'wildCardIndicator': True,
 'standingsDateTimeUtc': '2026-09-20T15:35:30Z',
 'standings': [{'clinchIndicator': 'p',
   'conferenceAbbrev': 'W',
   'conferenceHomeSequence': 1,
   'conferenceL10Sequence': 2,
   'conferenceName': 'Western',
   'conferenceRoadSequence': 1,
   'conferenceSequence': 1,
   'date': '2026-04-17',
   'divisionAbbrev': 'C',
   'divisionHomeSequence': 1,
   'divisionL10Sequence': 1,
   'divisionName': 'Central',
   'divisionRoadSequence': 1,
   'divisionSequence': 1,
   'gameTypeId': 2,
   'gamesPlayed': 82,
   'goalDifferential': 99,
   'goalDifferentialPctg': 1.207317,
   'goalAgainst': 203,
   'goalFor': 302,
   'goalsForPctg': 3.682927,
   'homeGamesPlayed': 41,
   'homeGoalDifferential': 49,
   'homeGoalsAgainst': 108,
   'homeGoalsFor': 157,
   'homeLosses': 9,
   'homeOtLosses': 6,
   'homePoints': 58,
   'homeRegulationPlusOtWins': 25,
   'homeRegulationWins': 25,
   'homeTies': 0,
   'homeWins': 26,
   'l10GamesPlayed': 10,
   'l10GoalDifferential': 14,

In [9]:
data['standings']

[{'clinchIndicator': 'p',
  'conferenceAbbrev': 'W',
  'conferenceHomeSequence': 1,
  'conferenceL10Sequence': 2,
  'conferenceName': 'Western',
  'conferenceRoadSequence': 1,
  'conferenceSequence': 1,
  'date': '2026-04-17',
  'divisionAbbrev': 'C',
  'divisionHomeSequence': 1,
  'divisionL10Sequence': 1,
  'divisionName': 'Central',
  'divisionRoadSequence': 1,
  'divisionSequence': 1,
  'gameTypeId': 2,
  'gamesPlayed': 82,
  'goalDifferential': 99,
  'goalDifferentialPctg': 1.207317,
  'goalAgainst': 203,
  'goalFor': 302,
  'goalsForPctg': 3.682927,
  'homeGamesPlayed': 41,
  'homeGoalDifferential': 49,
  'homeGoalsAgainst': 108,
  'homeGoalsFor': 157,
  'homeLosses': 9,
  'homeOtLosses': 6,
  'homePoints': 58,
  'homeRegulationPlusOtWins': 25,
  'homeRegulationWins': 25,
  'homeTies': 0,
  'homeWins': 26,
  'l10GamesPlayed': 10,
  'l10GoalDifferential': 14,
  'l10GoalsAgainst': 20,
  'l10GoalsFor': 34,
  'l10Losses': 2,
  'l10OtLosses': 1,
  'l10Points': 15,
  'l10RegulationPlus

In [18]:
records = []

for team_id, team in enumerate(data['standings'], start=1):
    team_abbrev = team['teamAbbrev']['default']
    team_name = team['teamName']['default']
    conf_name = team['conferenceName']
    div_name = team['divisionName']
    logo_url = team['teamLogo']

    records.append(dict(
        team_id=team_id,
        team_abbrev=team_abbrev,
        team_name=team_name,
        conference_name=conf_name,
        division_name=div_name,
        logo_url=logo_url
    ))

print(f'Total teams: {len(records)}')
records

Total teams: 32


[{'team_id': 1,
  'team_abbrev': 'COL',
  'team_name': 'Colorado Avalanche',
  'conference_name': 'Western',
  'division_name': 'Central',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg'},
 {'team_id': 2,
  'team_abbrev': 'CAR',
  'team_name': 'Carolina Hurricanes',
  'conference_name': 'Eastern',
  'division_name': 'Metropolitan',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/CAR_light.svg'},
 {'team_id': 3,
  'team_abbrev': 'DAL',
  'team_name': 'Dallas Stars',
  'conference_name': 'Western',
  'division_name': 'Central',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/DAL_light.svg'},
 {'team_id': 4,
  'team_abbrev': 'BUF',
  'team_name': 'Buffalo Sabres',
  'conference_name': 'Eastern',
  'division_name': 'Atlantic',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/BUF_light.svg'},
 {'team_id': 5,
  'team_abbrev': 'TBL',
  'team_name': 'Tampa Bay Lightning',
  'conference_name': 'Eastern',
  'division_name': 'Atlantic',
  'logo_url': 'https://ass

In [20]:
import mysql.connector

connection = mysql.connector.connect(
    host='localhost',
    port=3306,
    database='NHL_ANALYTICS_PROJECT',
    user='root',
    password="Omsivaya@20"
)

In [21]:
if connection.is_connected():
        cursor = connection.cursor()

In [22]:
%pip install mysql-connector-python


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS teams (
    team_id INT AUTO_INCREMENT PRIMARY KEY,
    team_abbrev VARCHAR(10) UNIQUE,
    team_name VARCHAR(100) NOT NULL,
    conference_name VARCHAR(100),
    division_name VARCHAR(100),
    logo_url TEXT
)
""")

connection.commit()
print('Teams table created successfully.')

Teams table created successfully.


In [31]:
insert_query = """
INSERT INTO teams (
    team_abbrev,
    team_name,
    conference_name,
    division_name,
    logo_url
)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_name = VALUES(team_name),
    conference_name = VALUES(conference_name),
    division_name = VALUES(division_name),
    logo_url = VALUES(logo_url)
"""

for i in records:
    team_values = (
        i['team_abbrev'],
        i['team_name'],
        i['conference_name'],
        i['division_name'],
        i['logo_url']
    )

    cursor.execute(insert_query, team_values)

connection.commit()
print(f"{len(records)} teams inserted or updated successfully.")

32 teams inserted or updated successfully.


In [35]:
import pandas as pd
df = pd.read_sql_query("select * from teams", connection)
df

/var/folders/f7/2dmww_657j51jk3c1xsbw30h0000gn/T/ipykernel_6132/2365296734.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("select * from teams", connection)


,team_id,team_abbrev,team_name,conference_name,division_name,logo_url
0,1,COL,Colorado Avalanche,Western,Central,https://assets.nhle.com/logos/nhl/svg/COL_ligh...
1,2,CAR,Carolina Hurricanes,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...
2,3,DAL,Dallas Stars,Western,Central,https://assets.nhle.com/logos/nhl/svg/DAL_ligh...
3,4,BUF,Buffalo Sabres,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BUF_ligh...
4,5,TBL,Tampa Bay Lightning,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/TBL_ligh...
5,6,MTL,Montréal Canadiens,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/MTL_ligh...
6,7,MIN,Minnesota Wild,Western,Central,https://assets.nhle.com/logos/nhl/svg/MIN_ligh...
7,8,BOS,Boston Bruins,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BOS_ligh...
8,9,OTT,Ottawa Senators,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/OTT_ligh...
9,10,PIT,Pittsburgh Penguins,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/PIT_ligh...


In [ ]:
#Last 3 tables

df_games = pd.read_json()

In [36]:
# Push it into SQL
!pip install sqlalchemy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 9.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [37]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
Note: you may need to restart the kernel to use updated packages.


In [38]:
import pymysql

from sqlalchemy import create_engine

engine = create_engine('mysql+pymysql://root:Omsivaya@20@localhost/NHL_ANALYTICS_PROJECT')

In [39]:
df_games = pd.read_json('/Users/nareshselvaraj/pandas course/NHL Analytics Hub Guvi Project/game_stats.json')
df_games

,stat_id,game_id,player_id,team_id,goals,assists,points,shots_on_goal,penalty_min,toi,plus_minus
0,1,2025010051,8480448,1,0,0,0,0,0,14:28,0
1,2,2025010051,8479525,1,0,0,0,0,0,18:48,-1
2,3,2025010051,8477492,1,0,1,1,0,0,18:57,1
3,6,2025010051,8477476,1,1,1,2,0,0,20:44,2
4,10,2025010051,8482947,1,1,0,1,0,0,12:48,1
...,...,...,...,...,...,...,...,...,...,...,...
47403,53923,2025021300,8476879,20,0,0,0,0,0,18:25,-1
47404,53924,2025021300,8476441,20,0,0,0,0,0,18:48,0
47405,53925,2025021300,8474563,20,0,1,1,0,0,22:20,1
47406,53926,2025021300,8479998,20,0,0,0,0,0,19:50,2


In [42]:
df_games.to_sql('game_stats',if_exists='replace', con=engine)

47408